- Añadimos un coste fijo diario por modelo, identificado como supuesto del caso de estudio

- Añadimos una variable binaria: modelo activado o desactivado

- Impedimos asignar consultas a un modelo desactivado

- Minimizamos costes por consulta + costes fijos

- Comparamos con el resultado básico y variamos costes fijos para observar cuánto conviene prescindir de algún modelo

In [1]:
from pathlib import Path
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

parameters = pd.read_csv(
    Path("data") / "processed" / "model_parameters.csv"
)

categories = parameters["category"].unique().tolist()
models = parameters["model"].unique().tolist()

cost = parameters.set_index(
    ["category", "model"]
)["mean_cost"].to_dict()

accuracy = parameters.set_index(
    ["category", "model"]
)["accuracy"].to_dict()

display(parameters)

,category,model,mean_cost,accuracy,sample_count
0,mathematics,gpt-3.5-turbo-1106,0.000105,0.392330,678
1,mathematics,gpt-4-1106-preview,0.001061,0.579646,678
2,mathematics,mistralai/mixtral-8x7b-chat,0.000063,0.221239,678
3,programming,gpt-3.5-turbo-1106,0.000338,0.641176,340
4,programming,gpt-4-1106-preview,0.009426,0.682353,340
5,programming,mistralai/mixtral-8x7b-chat,0.000176,0.535294,340
6,science,gpt-3.5-turbo-1106,0.000091,0.830782,1176
7,science,gpt-4-1106-preview,0.000923,0.960884,1176
8,science,mistralai/mixtral-8x7b-chat,0.000054,0.829932,1176


In [2]:
#Definimos el escenario y los nuevos costes fijos
#Son costes inventados que simulan el gasto por mantener disponible cada modelo
#durante ese día
demand = {
    "mathematics": 300,
    "programming": 300,
    "science": 400
}

base_capacity = {
    "gpt-3.5-turbo-1106": 600,
    "gpt-4-1106-preview": 250,
    "mistralai/mixtral-8x7b-chat": 700
}

minimum_accuracy = {
    "mathematics": 0.45,
    "programming": 0.60,
    "science": 0.85
}

# Synthetic daily fixed costs, in benchmark monetary units
fixed_cost = {
    "gpt-3.5-turbo-1106": 0.02,
    "gpt-4-1106-preview": 0.05,
    "mistralai/mixtral-8x7b-chat": 0.02
}

In [3]:
#Creamos una función que permite repetir experimentos sin copiar todo el código
#Añadimos una variable binaria active[model] que indica si decidimos activar ese modelo
#durante el día
    #1: está disponible para recibir consultas y se paga su coste fijo
    #0: está desactivado, no recibe consultas y no se paga su coste fijo

def solve_allocation(capacity, activation_cost):
    model = gp.Model("allocation_with_fixed_costs")
    model.Params.OutputFlag = 0
    model.Params.MIPGap = 0

    queries = model.addVars(
        categories, models,
        vtype=GRB.INTEGER,
        lb=0,
        name="queries"
    )

    active = model.addVars(
        models,
        vtype=GRB.BINARY,
        name="active"
    )

    variable_cost = gp.quicksum(
        cost[category, name] * queries[category, name]
        for category in categories
        for name in models
    )

    activation_total = gp.quicksum(
        activation_cost[name] * active[name]
        for name in models
    )

    model.setObjective(
        variable_cost + activation_total,
        GRB.MINIMIZE
    )

    for category in categories:
        model.addConstr(
            gp.quicksum(
                queries[category, name] for name in models
            ) == demand[category],
            name=f"demand_{category}"
        )

        model.addConstr(
            gp.quicksum(
                accuracy[category, name] * queries[category, name]
                for name in models
            ) >= minimum_accuracy[category] * demand[category],
            name=f"quality_{category}"
        )

    for name in models:
        model.addConstr(
            gp.quicksum(
                queries[category, name] for category in categories
            ) <= capacity[name] * active[name],
            name=f"capacity_{name}"
        )

    model.optimize()

    if model.Status != GRB.OPTIMAL:
        print(f"No optimal solution. Solver status: {model.Status}")
        return None

    allocation = pd.DataFrame(
        {
            name: [
                int(round(queries[category, name].X))
                for category in categories
            ]
            for name in models
        },
        index=categories
    )

    activation = pd.Series(
        {
            name: int(round(active[name].X))
            for name in models
        },
        name="active"
    )

    return {
        "allocation": allocation,
        "activation": activation,
        "variable_cost": variable_cost.getValue(),
        "fixed_cost": activation_total.getValue(),
        "total_cost": model.ObjVal
    }

In [4]:
#Resuelve y comprueba el escenario original con costes fijos
#Se muestran asignación, activación, calidad y costes y se comprueban las restricciones
base_result = solve_allocation(base_capacity, fixed_cost)

if base_result is not None:
    allocation = base_result["allocation"]

    display(allocation)

    model_summary = pd.DataFrame({
        "active": base_result["activation"],
        "assigned": allocation.sum(axis=0),
        "capacity": pd.Series(base_capacity)
    })
    display(model_summary)

    quality_rows = []

    for category in categories:
        assigned = allocation.loc[category].sum()

        expected_accuracy = sum(
            accuracy[category, name] * allocation.loc[category, name]
            for name in models
        ) / demand[category]

        quality_rows.append({
            "category": category,
            "assigned": assigned,
            "expected_accuracy": expected_accuracy,
            "minimum_accuracy": minimum_accuracy[category]
        })

        assert assigned == demand[category]
        assert expected_accuracy + 1e-6 >= minimum_accuracy[category]

    assert (
        model_summary["assigned"]
        <= model_summary["capacity"] * model_summary["active"]
    ).all()

    display(pd.DataFrame(quality_rows).set_index("category"))

    print(f"Variable cost: {base_result['variable_cost']:.6f}")
    print(f"Fixed cost: {base_result['fixed_cost']:.6f}")
    print(f"Total cost: {base_result['total_cost']:.6f}")


Restricted license - for non-production use only - expires 2027-11-29


,gpt-3.5-turbo-1106,gpt-4-1106-preview,mistralai/mixtral-8x7b-chat
mathematics,207,93,0
programming,184,0,116
science,0,62,338


,active,assigned,capacity
gpt-3.5-turbo-1106,1,391,600
gpt-4-1106-preview,1,155,250
mistralai/mixtral-8x7b-chat,1,454,700


,assigned,expected_accuracy,minimum_accuracy
category,,,
mathematics,300,0.450398,0.45
programming,300,0.600235,0.60
science,400,0.850230,0.85


Variable cost: 0.278522
Fixed cost: 0.090000
Total cost: 0.368522


In [5]:
#Se prueba un escenario donde activar modelos sea una decisión real
#Como experimento separado, aumentamos la capacidad de Mixtral a 1.000
#Así, GPT-4 y Mixtral pueden atender juntos la demanda. Variamos el coste fijo 
#de GPT-3.5 para comprobar si compensa mantenerlo disponible

alternative_capacity = base_capacity.copy()
alternative_capacity["mistralai/mixtral-8x7b-chat"] = 1000

experiment_rows = []

for gpt35_fixed_cost in [0.0, 0.02, 0.05, 0.10, 0.20]:
    scenario_fixed_cost = fixed_cost.copy()
    scenario_fixed_cost["gpt-3.5-turbo-1106"] = gpt35_fixed_cost

    result = solve_allocation(
        alternative_capacity,
        scenario_fixed_cost
    )

    if result is None:
        continue

    experiment_rows.append({
        "gpt35_fixed_cost": gpt35_fixed_cost,
        "gpt35_active": result["activation"]["gpt-3.5-turbo-1106"],
        "gpt35_queries": result["allocation"]["gpt-3.5-turbo-1106"].sum(),
        "gpt4_queries": result["allocation"]["gpt-4-1106-preview"].sum(),
        "mixtral_queries": result["allocation"][
            "mistralai/mixtral-8x7b-chat"
        ].sum(),
        "variable_cost": result["variable_cost"],
        "fixed_cost": result["fixed_cost"],
        "total_cost": result["total_cost"]
    })

fixed_cost_results = pd.DataFrame(experiment_rows)
display(fixed_cost_results)

,gpt35_fixed_cost,gpt35_active,gpt35_queries,gpt4_queries,mixtral_queries,variable_cost,fixed_cost,total_cost
0,0.00,1,391,155,454,0.278522,0.07,0.348522
1,0.02,1,391,155,454,0.278522,0.09,0.368522
2,0.05,1,391,155,454,0.278522,0.12,0.398522
3,0.10,1,391,155,454,0.278522,0.17,0.448522
4,0.20,1,391,155,454,0.278522,0.27,0.548522
